# 10 · Proof: every result equals V3

Compares V3-clean with V3's saved results, item by item. "Identical" means **exactly** equal (every probability to the last digit), not just close.

| Item | V3 file |
|---|---|
| Cells | `data/processed/cells_all.csv` |
| Outer-fold predictions of every run | `outputs/03_nested_cv/restricted/outer_predictions.csv` |
| Scores (mean, SD, min) | `outputs/04_results/aggregate/metrics_fold_mean_sd.csv` |
| Chosen model | `outputs/04_results/aggregate/selected_model.json` |
| Final model | `outputs/05_final/...` (frozen file) |
| XAI | `outputs/07_xai/...` |
| External tests | `outputs/05_final/aggregate/external_evaluation.json`, `outputs/09_new_external/aggregate/*.json` |
| Map | `outputs/10_map_model/restricted/siemreap_3d_points_120bh_50m.csv` |

V3's depth-only reference is left out (V3-clean does not use it).

**Run time:** 1-2 minutes.

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| V3 folder:", S["v3_root"])
import joblib
checks = []

def check(item, ok, detail=""):
    """ok = True / False, or None when the item was changed on purpose (final-model override)."""
    status = "CHANGED ON PURPOSE" if ok is None else ("IDENTICAL" if ok else "DIFFERENT")
    checks.append({"item": item, "status": status, "detail": detail})
    print(f"{status} | {item} {detail}")

cells = pd.read_csv(H.processed_dir(S) / "cells.csv")
V3_03 = H.v3_outputs(S, "03_nested_cv", "restricted")

## 1. Cells

In [ ]:
v3c = pd.read_csv(S["v3_root"] / "data" / "processed" / "cells_all.csv")
v3c = v3c[v3c.is_downtown & v3c.set.isin(["development", "external"])].reset_index(drop=True)
cols = ["BH_Code", "cell", "depth_top_m", "UTM_Easting_m", "UTM_Northing_m", "Top_Elevation_m", "y", "measured", "set"]
check("cells", len(cells) == len(v3c) and all((cells[c].values == v3c[c].values).all() for c in cols), f"({len(cells)} cells)")

## 2. Outer-fold predictions of every run and fold

In [ ]:
mine = pd.read_csv(H.out_dir(S, "03_spatial_cv", private=True) / "outer_predictions.csv")
v3p = pd.read_csv(V3_03 / "outer_predictions.csv")
rows = []
for key, a in mine.groupby(["model", "block_m", "purge_m", "outer_fold"]):
    m, b, p, k = key
    v = v3p[(v3p.model == m) & (v3p.block_m == b) & (v3p.purge_m == p) & (v3p.outer_fold == k)]
    same = len(a) == len(v) and (a.BH_Code.values == v.BH_Code.values).all() and (a.cell.values == v.cell.values).all() \
        and np.array_equal(a.p_clay.values, v.p_clay.values) and np.array_equal(a.y_pred.values, v.y_pred.values) \
        and np.array_equal(a.threshold.values, v.threshold.values)
    diff = np.abs(a.p_clay.values - v.p_clay.values).max() if len(a) == len(v) else np.nan
    rows.append({"model": m, "block_m": b, "purge_m": p, "fold": k, "cells": len(a), "identical": same, "max_diff_p_clay": diff})
pt = pd.DataFrame(rows)
display(pt.pivot_table(index=["block_m", "purge_m", "model"], columns="fold", values="identical", aggfunc="first"))
check("outer-fold predictions", pt.identical.all(), f"({int(pt.identical.sum())} of {len(pt)} run-folds)")

## 3. Scores and the chosen model

In [ ]:
s_mine = pd.read_csv(H.out_dir(S, "04_results") / "metrics_fold_mean_sd.csv")
s_v3 = pd.read_csv(H.v3_outputs(S, "04_results") / "metrics_fold_mean_sd.csv")
keys = ["scenario", "block_m", "purge_m", "model"]
j = s_mine.merge(s_v3, on=keys, suffixes=("", "_v3"))
num = [c for c in s_mine.columns if c not in keys and c in s_v3.columns]
worst = max((j[c] - j[c + "_v3"]).abs().max() for c in num)
check("scores (mean, SD, min of every metric)", len(j) == len(s_mine) and worst == 0, f"(largest difference {worst})")
main_ = j[(j.block_m == H.block_sizes(S)[0]) & (j.purge_m == 0)][["model", "macro_f1_mean", "macro_f1_mean_v3"]]
display(main_.round(4))
sel_mine = json.loads((H.out_dir(S, "04_results") / "selected_model.json").read_text())["selected"]
sel_v3 = json.loads((H.v3_outputs(S, "04_results") / "selected_model.json").read_text())["selected"]["model"]
check("chosen model", sel_mine == sel_v3, f"({sel_mine} vs V3 {sel_v3})")

## 4. Final model

In [ ]:
rec = json.loads((H.out_dir(S, "05_final_model") / "final_model.json").read_text())
v3r = json.loads((H.v3_outputs(S, "05_final") / "final_model.json").read_text())
SAME_FINAL = rec["model"] == v3r["model"]
if SAME_FINAL:
    check("final model parameters and threshold", rec["params"] == v3r["params"] and rec["threshold"] == v3r["threshold"],
          f"(threshold {rec['threshold']} vs {v3r['threshold']})")
    m1 = joblib.load(H.out_dir(S, "05_final_model", private=True) / rec["model_file"])
    m2 = H.load_v3_model(S, H.v3_outputs(S, "05_final", "restricted") / v3r["model_file"])
    X = cells[H.FEATURES].values
    check("final model predictions (all 120 boreholes)", np.array_equal(m1.predict_proba(X)[:, 1], m2.predict_proba(X)[:, 1]))
else:
    check("final model", None, f"({rec['model']} by override; the rule and V3 chose {v3r['model']}. Reason: {rec.get('override_reason')})")

## 5. Explainable AI

In [ ]:
pm = pd.read_csv(H.out_dir(S, "06_xai") / "permutation_importance_by_fold.csv")
pv = pd.read_csv(H.v3_outputs(S, "07_xai") / "permutation_importance_by_fold.csv")
pj = pm.merge(pv, on=["model", "fold", "cells", "input"], suffixes=("", "_v3"))
check("permutation importance", len(pj) == len(pm) and (pj.drop_mean - pj.drop_mean_v3).abs().max() == 0,
      f"({len(pj)} values)")
sm = pd.read_csv(H.out_dir(S, "06_xai", private=True) / "shap_values_by_cell.csv")
sv = pd.read_csv(H.v3_outputs(S, "07_xai", "restricted") / "shap_values_by_cell.csv")
sv = sv[sv.model.isin(sm.model.unique())].reset_index(drop=True)
scol = ["shap_Easting", "shap_Northing", "shap_Elevation", "base"]
check("SHAP values", len(sm) == len(sv) and all(np.array_equal(sm[c].values, sv[c].values) for c in scol),
      f"({len(sm)} cells)")

## 6. External tests

In [ ]:
ext = pd.read_csv(H.out_dir(S, "07_external_tests") / "external_tests.csv", float_precision="round_trip")   # exact, to compare with V3's JSON
records = {"A. V3 holdout": H.v3_outputs(S, "05_final") / "external_evaluation.json",
           "B. new boreholes": H.v3_outputs(S, "09_new_external") / "new_external_evaluation.json",
           "C. new boreholes, SS gaps filled": H.v3_outputs(S, "09_new_external") / "new_external_evaluation_ssfill.json"}
mets = ["n", "accuracy", "balanced_accuracy", "macro_f1", "roc_auc", "brier"]
for test, path in records.items():
    v = pd.DataFrame(json.loads(path.read_text())["metrics"])
    v = v[v.model == v3r["model"]]                               # V3 scored its final model only
    a = ext[(ext.test == test) & (ext.model == v3r["model"])]
    jj = a.merge(v, on=["cells", "group"], suffixes=("", "_v3"))
    worst = max((jj[c] - jj[c + "_v3"]).abs().max() for c in mets)
    check(f"external test {test} ({v3r['model']})", len(jj) == len(a) > 0 and worst == 0, f"({len(jj)} rows, largest difference {worst})")

## 7. Map

In [ ]:
spacing = int(S["quick"]["map_spacing_m"] if S["run"]["quick"] else S["map"]["spacing_m"])
gm = pd.read_csv(H.out_dir(S, "08_map", private=True) / f"siemreap_3d_points_120bh_{spacing}m.csv")
gv = pd.read_csv(H.v3_outputs(S, "10_map_model", "restricted") / f"siemreap_3d_points_120bh_{spacing}m.csv")
if SAME_FINAL:
    same = len(gm) == len(gv) and all(np.array_equal(gm[c].values, gv[c].values) for c in ["X", "Y", "Z", "p_clay", "clay", "reliability_zone"])
    check("3D map (every cell: X, Y, Z, P(Clay), class, reliability zone)", same, f"({len(gm):,} cells)")
else:
    same_grid = len(gm) == len(gv) and all(np.array_equal(gm[c].values, gv[c].values) for c in ["X", "Y", "Z", "reliability_zone"])
    check("3D map grid (X, Y, Z, reliability zone)", same_grid, f"({len(gm):,} cells)")
    check("3D map P(Clay) and class", None, f"(made with {rec['model']}; V3's map used {v3r['model']}; "
          f"cells with the same class: {np.mean(gm.clay.values == gv.clay.values):.1%})")

## Summary

In [ ]:
summary = pd.DataFrame(checks)
display(summary)
summary.to_csv(H.out_dir(S, "10_compare_with_v3") / "comparison_with_v3.csv", index=False)
if (summary.status == "DIFFERENT").any():
    print("\nSOME RESULTS DIFFER - see the table above")
elif (summary.status == "CHANGED ON PURPOSE").any():
    print("\nEVERYTHING IDENTICAL TO V3, except the items changed on purpose (final-model override)")
else:
    print("\nALL RESULTS IDENTICAL TO V3")